# Battery aging — how a cell wears out

**A lab notebook for the "normal aging" section of a battery-modelling course.**

Cell performance deteriorates gradually because of unwanted chemical side reactions and physical
changes to the active materials. Aging is essentially irreversible, and it eventually ends in cell
failure. This notebook takes that statement apart.

Each section has three parts, in the same order every time:

1. **What happens** — the mechanism in plain language.
2. **A panel you can drive** — one focused interactive figure. Move a slider, watch the physics.
3. **The model in Python** — the same equations the panel uses, as code you can read and change.

Every section ends with the papers it is drawn from, and §12 carries the full bibliography with DOI
links. Section 10 is a one-page recap table.

**Running it.** Run all cells top to bottom (*Kernel → Restart & Run All*). There is nothing to
install: the panels are self-contained HTML and the Python uses only the standard library. They work
in JupyterLab, classic Notebook and VS Code alike.

> The equations are teaching models — a √t film, an Arrhenius factor, one-line stress terms. They
> reproduce the direction, the shape and the rough relative size of each effect. They are not a
> state-of-health estimator, and no number here should be quoted for a real cell.

In [ ]:
# ── Setup: load the interactive panels ───────────────────────────────────────
# The panels are self-contained HTML documents stored compressed in this cell.
# panel(name) decompresses one and embeds it in an isolated frame, so its styling
# can never leak into the notebook and the notebook's can never leak into it.
import base64
import json
import zlib
from html import escape

from IPython.display import HTML

_ASSETS = json.loads(zlib.decompress(base64.b64decode("...")).decode())

def panel(name, height=440):
    """Embed one interactive panel."""
    doc = ("<!doctype html><html lang='en' data-theme='light'><head><meta charset='utf-8'>"
           "<meta name='viewport' content='width=device-width, initial-scale=1'>"
           "<style>" + _ASSETS["css"] + "</style></head><body>"
           "<script>" + _ASSETS["lib"] + "</script>" + _ASSETS["widgets"][name] + "</body></html>")
    return HTML(
        f'<iframe class="aging-panel" srcdoc="{escape(doc, quote=True)}" title="{name} panel" '
        f'style="width:100%;height:{height}px;border:0;display:block;color-scheme:light dark"></iframe>')

print(f"{len(_ASSETS['widgets'])} panels ready · no external dependencies")

Set light or dark once here; every panel below follows.

In [ ]:
panel("theme", 64)

## 1 · The whole chain on one page

Six mechanisms. Three effects. One outcome.

The mechanisms on the left are side reactions and physical changes — and not every chemistry suffers
from all of them. Whatever the cause, it is expressed as one or more of exactly three things:
**impedance rises**, **capacity falls**, **self-discharge rises**. Nothing else.

Select a mechanism to trace where it ends up.

*Where this comes from — [Vetter et al. 2005](https://doi.org/10.1016/j.jpowsour.2005.01.006) (the canonical taxonomy of these mechanisms) · [Edge et al. 2021](https://doi.org/10.1039/D1CP00359C) (how they interact) · [Birkl et al. 2017](https://doi.org/10.1016/j.jpowsour.2016.12.011) (why they all collapse into three measurable effects). Full citations in §12.*

In [ ]:
panel("map", 470)

## 2 · Where it happens

This is a zoomed cross-section of **one repeating electrode sandwich** — copper current collector,
porous graphite negative electrode, separator, porous NMC positive electrode, aluminium collector.
Electrolyte fills every pore of all three middle layers.

The single most common misconception is that these mechanisms happen in separate places. They do
not. They happen on particle surfaces, inside particles, and at material interfaces, distributed
throughout the whole cell. In a pouch cell these sandwiches are stacked; in a cylindrical cell they
are wound.

Pick a duty, set the temperature and the elapsed service life, then click a numbered marker.

*Where this comes from — Plett 2015, Vol. I (the course text) for the porous-electrode picture; [Vetter et al. 2005](https://doi.org/10.1016/j.jpowsour.2005.01.006) for where each mechanism sits in it; geometry follows [Argonne's cell description](https://www.anl.gov/science-101/batteries). Full citations in §12.*

In [ ]:
panel("cell", 640)

### The rate law underneath all of it

Almost every mechanism in this notebook is a chemical reaction, and chemical reactions obey
Arrhenius:

$$\frac{k(T)}{k(T_\text{ref})} = \exp\!\left[\frac{E_a}{R}\left(\frac{1}{T_\text{ref}} - \frac{1}{T}\right)\right]$$

For a typical activation energy of ~50 kJ/mol this is the familiar rule of thumb: **roughly double
the rate for every +10 °C**. Heat does not introduce a new kind of damage — it runs the same damage
faster.

*Where this comes from — [Broussely et al. 2005](https://doi.org/10.1016/j.jpowsour.2005.03.172) measured the calendar-aging rates the Arrhenius factor is fitted to; [Waldmann et al. 2014](https://doi.org/10.1016/j.jpowsour.2014.03.112) mapped how the dominant mechanism itself changes with temperature. Full citations in §12.*

In [ ]:
import math

R_GAS = 8.314  # J / (mol·K)

def arrhenius(T_C, Ea, T_ref_C=25.0):
    """How many times faster a thermally activated reaction runs at T than at T_ref."""
    return math.exp(Ea / R_GAS * (1 / (T_ref_C + 273.15) - 1 / (T_C + 273.15)))

print("Reaction rate relative to 25 °C\n")
print(f"{'T (°C)':>8}  {'Ea = 45 kJ/mol':>15}  {'50 (SEI)':>10}  {'62 (gassing)':>13}")
for T in (-10, 0, 10, 25, 35, 45, 60):
    print(f"{T:>8}  {arrhenius(T, 45_000):>14.2f}×  {arrhenius(T, 50_000):>9.2f}×  {arrhenius(T, 62_000):>12.2f}×")

## 3 · Passivation — the SEI film

> *Lithium-ion cells with graphitic negative electrodes are prone to a particular kind of corrosion
> known as passivation.*

The solvent in the electrolyte is not chemically stable at the voltages found in most lithium-ion
cells, so it reacts with the graphite particles. The reaction products form a layer on the particle
surfaces — the **solid–electrolyte interphase**.

That film is not purely bad news. It *protects* the graphite from further reaction, which is why the
process slows so dramatically after the first few cycles. But it never stops: SEI keeps growing
slowly for the cell's entire life, and every bit of growth consumes lithium that can no longer be
cycled.

Two ideas do most of the work here, and between them they explain most of what a battery does as it
ages:

- **√t growth.** The film is its own diffusion barrier, so thickness grows like `δ ∝ √t`, not linearly.
- **Arrhenius.** Warmer means faster, roughly doubling per +10 °C.

Set the temperature to 45 °C and watch what a hot summer does to a cell that is doing nothing at all.

*Where this comes from — [Pinson & Bazant 2013](https://doi.org/10.1149/2.044302jes) derive the √t law and its lifetime consequences · [Broussely et al. 2005](https://doi.org/10.1016/j.jpowsour.2005.03.172) for the storage measurements · [Vetter et al. 2005](https://doi.org/10.1016/j.jpowsour.2005.01.006) §2 for the film chemistry. Full citations in §12.*

In [ ]:
panel("sei", 440)

In [ ]:
SEI_D0    = 12.0    # nm of film after one year at 25 °C
SEI_EA    = 50_000  # J/mol
LI_PER_NM = 0.0043  # fraction of cyclable lithium consumed per nm of film

def sei_thickness(years, T_C):
    """Film thickness in nm — diffusion-limited, so it grows with the square root of time."""
    return SEI_D0 * math.sqrt(max(years, 0) * arrhenius(T_C, SEI_EA))

def sei_capacity(years, T_C):
    """Capacity remaining once the film has locked up that much lithium."""
    return max(0.0, 1 - LI_PER_NM * sei_thickness(years, T_C))

print("Capacity remaining, storage only, no cycling\n")
print(f"{'years':>6}  " + "  ".join(f"{T:>3} °C" for T in (0, 25, 45)))
for y in (1, 2, 5, 10):
    print(f"{y:>6}  " + "  ".join(f"{100*sei_capacity(y, T):>5.1f}%" for T in (0, 25, 45)))

# The first year costs more than the next four put together — that is the √t curve.
print(f"\nfilm after 1 year at 25 °C: {sei_thickness(1, 25):.0f} nm")
print(f"film after 4 years:         {sei_thickness(4, 25):.0f} nm   (4× the time, 2× the film)")

## 4 · Gas generation

> *Some chemistries naturally produce gaseous products when charging, which ideally return to their
> prior aqueous state when discharging.*

Two quite different failure paths come out of one reaction:

- **The gas escapes.** Through a breach in the enclosure, that capacity is gone permanently. In many
  cases it is also dangerous — in some cells the released gases are explosive. Lead-acid cells give
  off oxygen and hydrogen when overcharged.
- **The gas stays.** In a sealed cell pressure builds until the cell ruptures or explodes, unless it
  has a **release vent**. Vents make the failure survivable; they do not make it free, because
  venting loses electrolyte too.

Push the overcharge slider and watch the two thresholds arrive in order: the cell starts to swell,
then the vent opens.

*Where this comes from — [Rowden & Garcia-Araez 2020](https://doi.org/10.1016/j.egyr.2020.02.022), a review of which gases form, when, and what they do · [Vetter et al. 2005](https://doi.org/10.1016/j.jpowsour.2005.01.006) for electrolyte decomposition. Full citations in §12.*

In [ ]:
panel("gas", 480)

In [ ]:
GAS_G0    = 1.20    # % of free volume generated per year, at 25 °C, within spec
GAS_R     = 0.90    # 1/year — how fast gas is reabsorbed on discharge
SWELL, VENT = 5.0, 14.0   # % of free volume

def gas_volume(years, T_C, overcharge=1.0):
    """Generation minus recombination:  V(t) = (g/r)·(1 − e^(−r·t)).

    In spec the two balance and the gas plateaus. Push generation up with heat or
    overcharge and the plateau climbs past swelling, then past the vent.
    """
    g = GAS_G0 * arrhenius(T_C, 62_000) * overcharge
    return g / GAS_R * (1 - math.exp(-GAS_R * max(years, 0)))

print("Steady-state gas held, % of the cell's free volume\n")
print(f"{'':>12}" + "".join(f"{f'×{oc:.0f} charge':>14}" for oc in (1, 2, 3, 4)))
for T in (25, 40, 55):
    row = [gas_volume(20, T, oc) for oc in (1, 2, 3, 4)]
    flag = ["", " swell", " VENT"]
    print(f"{T:>9} °C  " + "".join(
        f"{v:>9.1f}%{flag[(v >= SWELL) + (v >= VENT)]:<5}" for v in row))

## 5 · Crystal formation

> *When material is removed from an electrode during discharge, it will not generally return to the
> same location when the cell is recharged. Instead, crystal structures will tend to form on the
> electrode surfaces.*

This is a very general redox-cell mechanism, and its consequence is geometric rather than chemical.
A porous electrode works *because* it has an enormous internal surface area — that is where the
reaction happens. As crystals build up, the effective surface area shrinks and the reaction has to
squeeze through a smaller doorway.

Note what that costs you: resistance rises, so the cell loses its ability to deliver **high power**.
This is a power problem before it is a capacity problem.

*Where this comes from — [Vetter et al. 2005](https://doi.org/10.1016/j.jpowsour.2005.01.006) on surface-film and morphology changes · [Han et al. 2019](https://doi.org/10.1016/j.etran.2019.100005) on how loss of active surface shows up as power fade. Full citations in §12.*

In [ ]:
panel("crystal", 440)

In [ ]:
OCV, R0, V_CUT = 3.7, 0.030, 2.8   # V, Ω, V — one illustrative cell

def area_ratio(cycles, susceptibility=1.0):
    """Fraction of the original active surface area still exposed."""
    return 1 / (1 + 0.0016 * susceptibility * max(cycles, 0) ** 0.85)

def peak_power(resistance):
    """Most power the cell can deliver before it hits its cut-off voltage."""
    i_max = (OCV - V_CUT) / resistance
    return i_max * V_CUT, i_max

print("Power capability as crystals accumulate\n")
print(f"{'cycles':>7}  {'area':>7}  {'resistance':>11}  {'peak power':>11}  {'max current':>12}")
for n in (0, 250, 500, 1000, 2000):
    a = area_ratio(n)
    r = R0 / a                      # a smaller doorway is a bigger resistance
    p, i = peak_power(r)
    print(f"{n:>7}  {100*a:>6.0f}%  {1000*r:>9.0f} mΩ  {p:>9.0f} W  {i:>10.0f} A")

## 6 · Dendrites — crystal growth that becomes a safety problem

> *These treelike structures can grow through the separator, causing an increase in the cell's
> self-discharge rate or even a short circuit.*

In lithium-ion cells the trigger is specific and well understood: **low-temperature operation or
overcurrent during charging** deposits lithium metal on the negative-electrode particles instead of
inserting it into them, and those deposits grow as dendrites.

Why cold makes it worse is worth stating plainly. Intercalation is a *kinetic* process, so its rate
falls with temperature like every other reaction. The charging current, however, is whatever the
charger decides to push. When lithium arrives faster than the graphite can take it in, it has
nowhere to go but onto the surface — as metal.

The heat map is the practical summary: the danger is the **top-left corner**, high current and low
temperature. It is why a well-designed battery management system refuses to fast-charge a cold pack,
and warms it first if it can.

*Where this comes from — [Waldmann, Hogg & Wohlfahrt-Mehrens 2018](https://doi.org/10.1016/j.jpowsour.2018.02.063), the review this section is built on, including the temperature and rate dependence · [Waldmann et al. 2014](https://doi.org/10.1016/j.jpowsour.2014.03.112) for the post-mortem evidence that plating dominates below ~25 °C. Full citations in §12.*

In [ ]:
panel("plating", 460)

In [ ]:
def plating_risk(T_C, c_rate):
    """WILL it plate? A probability, so it saturates at 1."""
    if c_rate <= 0:
        return 0.0
    drive = c_rate / arrhenius(T_C, 35_000)      # charge-transfer kinetics slow in the cold
    return max(0.0, min(1.0, (drive - 0.35) / (drive + 1.0)))

def plating_drive(T_C, c_rate):
    """HOW MUCH plates? An excess, so it does not saturate.

    The charger pushes the current it was told to; the graphite absorbs what its
    kinetics allow. The difference has nowhere to go but onto the surface, and it
    keeps growing as the cell gets colder. This is why cold charging has no floor.
    """
    if c_rate <= 0:
        return 0.0
    return max(0.0, c_rate / arrhenius(T_C, 35_000) - 0.6)

print("Charging at 2 C\n")
print(f"{'T (°C)':>7}  {'kinetics':>9}  {'will it plate':>14}  {'how much':>9}")
for T in (-10, 0, 10, 25, 40):
    print(f"{T:>7}  {arrhenius(T, 35_000):>8.2f}×  {100*plating_risk(T, 2.0):>13.0f}%  {plating_drive(T, 2.0):>9.1f}")

print("\nSame cell at 25 °C, varying the charge rate")
for c in (0.3, 1.0, 2.0, 3.0):
    print(f"  {c:>4.1f} C   {100*plating_risk(25, c):>3.0f}% likely   drive {plating_drive(25, c):.1f}")

## 7 · Volume change, cracking and loss of contact

> *Charging and discharging intercalation-based electrodes causes volume changes, which stress the
> electrodes and can lead to cracking of the active materials.*

This is the purely **mechanical** mechanism, and the one that responds to *how* you cycle rather
than to temperature. Every insertion and removal of lithium swells and shrinks the host lattice; do
it a few thousand times and the particle behaves like any other fatigued material.

It damages the cell in two ways at once:

- **Fracture and structural collapse** take active material out of service → capacity fade. In some
  positive-electrode materials the structure collapses so completely that it can no longer
  intercalate lithium at all.
- **Broken binder and conductive additives** electrically orphan particles that are still chemically
  fine → impedance rise.

Shallow cycles are dramatically gentler than deep ones. Note the log scale on the right — that is
why keeping a pack between roughly 30 % and 70 % state of charge buys so much cycle life.

*Where this comes from — [Christensen & Newman 2006](https://doi.org/10.1007/s10008-006-0095-1) on stress generation and fracture during intercalation · [Edge et al. 2021](https://doi.org/10.1039/D1CP00359C) on particle cracking and contact loss as coupled degradation modes. Full citations in §12.*

In [ ]:
panel("cracking", 460)

In [ ]:
def mech_damage(cycles, dod):
    """Accumulated mechanical damage, 0 (pristine) → 1 (thoroughly fractured)."""
    return min(1.0, 0.0095 * (max(cycles, 0) * dod ** 1.6) ** 0.6)

def mech_capacity(cycles, dod):
    return 1 - 0.45 * mech_damage(cycles, dod)        # active material lost

def mech_resistance(cycles, dod):
    return 1 + 1.60 * mech_damage(cycles, dod)        # contacts broken

def cycles_to_80(dod):
    n = 10
    while n <= 40_000:
        if mech_capacity(n, dod) <= 0.80:
            return n
        n += 10
    return None

print("Cycle life against depth of discharge\n")
print(f"{'DOD':>5}  {'cycles to 80 %':>15}  {'equivalent full cycles':>23}")
for dod in (0.2, 0.4, 0.6, 0.8, 1.0):
    n = cycles_to_80(dod)
    print(f"{100*dod:>4.0f}%  {n:>15,}  {n*dod:>23,.0f}")
print("\nHalving the depth of discharge buys far more than halving the number of cycles.")

## 8 · The whole cell — all six mechanisms at once

Everything above, running together. The stacked bars are the interesting part: change the duty and
watch which mechanism is actually to blame change with it.

One caveat the panel cannot show you. Capacity fade is not perfectly monotonic in the short term:
in some chemistries a little capacity can be clawed back by **reconditioning** — subjecting the cell
to one or more deep discharges. The general trend is still downward, and none of the underlying
damage is undone.

Three results worth pulling out by hand:

- **Shelf storage at 25 °C** is dominated by passivation, and takes over a decade to reach 80 %.
- **Hot operation** compresses that to a couple of years, without the cell doing anything different.
- **Cold fast-charging** hands the blame to plating instead — a different mechanism, a different
  warning sign, and a safety problem rather than a performance one.

*Where this comes from — [Birkl et al. 2017](https://doi.org/10.1016/j.jpowsour.2016.12.011) on separating the modes from measurable behaviour · [Han et al. 2019](https://doi.org/10.1016/j.etran.2019.100005) for whole-life-cycle behaviour · [Attia et al. 2022](https://doi.org/10.1149/1945-7111/ac6d13) on why the trajectory can bend sharply downward. Full citations in §12.*

In [ ]:
panel("sim", 620)

In [ ]:
# The same model the panel runs, term for term.
DUTY = {                       # c_rate, cycling intensity, depth of discharge, gas ×, corrosion ×
    "shelf storage":      dict(c_rate=0.0, cyc=0.02, dod=0.0, gas=1.0, cor=1.0, T=25),
    "hot operation":      dict(c_rate=0.7, cyc=0.60, dod=0.5, gas=1.4, cor=1.5, T=45),
    "cold / fast charge": dict(c_rate=2.5, cyc=0.80, dod=0.6, gas=1.0, cor=1.0, T=0),
    "deep cycling":       dict(c_rate=1.0, cyc=1.00, dod=0.9, gas=1.1, cor=1.1, T=30),
    "overcharge":         dict(c_rate=1.0, cyc=0.50, dod=0.5, gas=4.0, cor=2.6, T=40),
}
CAP_W = dict(sei=0.30, dendrite=0.34, cracking=0.26, gas=0.10, crystal=0.06)   # → capacity fade
IMP_W = dict(sei=0.55, corrosion=1.60, cracking=1.30, crystal=0.90, gas=0.25)  # → impedance rise
SD_W  = dict(dendrite=3.40, gas=1.10, crystal=0.50)                            # → self-discharge

def severities(T_C, years, d):
    """How far along each mechanism is, 0 → 1."""
    t, cl = max(years, 0), lambda v: min(1.0, max(0.0, v))
    return dict(
        sei       = cl(0.160 * math.sqrt(t * arrhenius(T_C, 50_000)) * (1 + 0.30 * d["cyc"])),
        corrosion = cl(0.032 * t * arrhenius(T_C, 45_000) * d["cor"]),
        gas       = cl(0.013 * t * arrhenius(T_C, 62_000) * d["gas"]),
        crystal   = cl(0.100 * (t * (0.25 + d["cyc"])) ** 0.6),
        cracking  = cl(0.105 * (t * d["cyc"] * (0.35 + d["dod"])) ** 0.7),
        dendrite  = cl(0.115 * plating_drive(T_C, d["c_rate"]) * math.sqrt(t * (0.15 + d["cyc"]) / 10)),
    )

def effects(s):
    """The only three things any of it can turn into."""
    return dict(
        capacity       = 1 - min(0.75, sum(CAP_W[k] * s[k] for k in CAP_W)),
        impedance      = 1 + sum(IMP_W[k] * s[k] for k in IMP_W),
        self_discharge = 1.8 * (1 + sum(SD_W[k] * s[k] for k in SD_W)),
    )

def years_to_80(T_C, d):
    t = 0.1
    while t <= 30:
        if effects(severities(T_C, t, d))["capacity"] <= 0.80:
            return t
        t += 0.1
    return None

print(f"{'duty':<20}{'T':>5}  {'capacity @5y':>13}  {'impedance':>10}  {'self-disch':>11}  {'to 80 %':>8}  {'blame':>10}")
for name, d in DUTY.items():
    s = severities(d["T"], 5, d)
    e = effects(s)
    eol = years_to_80(d["T"], d)
    blame = max(CAP_W, key=lambda k: CAP_W[k] * s[k])
    print(f"{name:<20}{d['T']:>4}°  {100*e['capacity']:>12.1f}%  {e['impedance']:>9.2f}×"
          f"  {e['self_discharge']:>8.1f} %/mo  {(f'{eol:.1f} y' if eol else '>30 y'):>8}  {blame:>10}")

## 9 · Temperature is the master variable

> *Aging processes are generally accelerated by elevated temperatures. The best way to extend a
> cell's life is to maintain its temperature in an acceptable range.*

That is the single most actionable sentence in the whole topic, and the panel below is why.

The catch — and the reason the total curve is **U-shaped** rather than simply rising — is lithium
plating. Cooling a cell slows the side reactions, but it also slows the intercalation you *want*, so
a cold cell that is being charged plates lithium instead. **Cold is only free if no current is
flowing.** Switch between *shelf storage* and *cold / fast charge* to see both halves.

Both curves in the panel are derived from the model in section 8, not written separately, so the
U-shape is a result rather than a second opinion.

*Where this comes from — [Waldmann et al. 2014](https://doi.org/10.1016/j.jpowsour.2014.03.112) is the key result: post-mortem analysis of cells aged from −20 to +70 °C, showing two competing regimes with a minimum near room temperature · [Broussely et al. 2005](https://doi.org/10.1016/j.jpowsour.2005.03.172) for the thermal half. Full citations in §12.*

In [ ]:
panel("window", 520)

In [ ]:
print("Years until 80 % capacity, by temperature\n")
print(f"{'duty':<20}" + "".join(f"{T:>8}°C" for T in (-10, 0, 10, 25, 40, 55)))
for name, d in DUTY.items():
    row = []
    for T in (-10, 0, 10, 25, 40, 55):
        e = years_to_80(T, d)
        row.append(f"{e:>8.1f}  " if e else f"{'>30':>8}  ")
    print(f"{name:<20}" + "".join(row))

print("\nWhere each duty ages slowest (searching −20 to 60 °C):\n")
for name, d in DUTY.items():
    lives = [(years_to_80(T, d) or 30.0, T) for T in range(-20, 61)]
    top = max(life for life, _ in lives)
    band = [T for life, T in lives if life >= top - 1e-9]
    where = f"{band[0]} °C" if len(band) == 1 else f"{band[0]} to {band[-1]} °C"
    life = "≥ 30 y (model cap)" if top >= 30 else f"{top:.1f} y"
    print(f"  {name:<20} {where:<18} {life}")

# With no current flowing there is no plating penalty, so storage simply keeps
# getting slower as the cell gets colder — the flat band above is the model's cap,
# not a physical optimum.

## 10 · Recap

| Mechanism | Where it lives | Feeds mainly | Accelerated by | Caveat |
|---|---|---|---|---|
| **01 Corrosion** | collectors, active *and* inactive materials | impedance ↑ | temperature | a catch-all term; not every chemistry suffers every reaction |
| **02 Passivation (SEI)** | graphite particle surfaces | capacity ↓ | temperature, time | specific to graphitic negative electrodes |
| **03 Gas generation** | pores, then the cell's free volume | capacity ↓, safety | temperature, overcharge | species and rates depend on chemistry, SOC and abuse history |
| **04 Crystal formation** | electrode surfaces | impedance ↑ → power fade | cycling | very general to redox cells; morphology varies enormously |
| **05 Dendrites** | negative electrode → through the separator | self-discharge ↑, shorts | cold charging, high current | a risk condition, not normal operation |
| **06 Volume change** | inside particles, binder, additives | capacity ↓ **and** impedance ↑ | deep cycling | severity depends on chemistry, particle size, electrode design |

Three sentences worth keeping:

1. Aging is **gradual and essentially irreversible**, and it ends in failure. Reconditioning recovers
   a little in some chemistries; it does not reverse the trend.
2. Everything arrives as **impedance ↑, capacity ↓, self-discharge ↑** — nothing else.
3. **Temperature is the master variable**, with cold charging as the one important exception.

## 11 · Check yourself

Work these out from the panels above before reading the answers.

<details><summary><b>1.</b> The SEI film is a degradation product. Why is a cell with no SEI at all worse off?</summary>

Because the film is what *stops* the reaction. Without a passivating layer the solvent would keep
reacting with the graphite at its initial, very fast rate. The SEI is the reason the loss curve bends
over into √t instead of continuing as a straight line. A self-limiting side reaction is enormously
better than an unlimited one. In section 3, compare the film after 1 year with the film after 4.
</details>

<details><summary><b>2.</b> A pack is fast-charged at 0 °C every day. Which mechanism dominates, and which effect shows up first?</summary>

Lithium plating. In section 6, set 0 °C and 2–3 C: the propensity is up near 80 %, and the drive —
how much actually plates — is several times its 25 °C value. The first symptom is usually
**increased self-discharge**, because the deposits reach into the separator long before they bridge
it, together with capacity loss, since plated lithium is often electrically isolated and never comes
back. Impedance rise from this mechanism arrives later.
</details>

<details><summary><b>3.</b> Two identical cells sit on a shelf, one at 25 °C and one at 45 °C. Roughly how much faster does the hot one age?</summary>

About 3.5× faster, from the Arrhenius factor for a ~50 kJ/mol reaction — doubling per 10 °C, and
20 °C is two doublings plus a bit. In section 8, set *shelf storage* and compare the two: time to
80 % goes from over a decade to a couple of years, with the cell doing no work at all.
</details>

<details><summary><b>4.</b> Why is a rising self-discharge rate treated as more alarming than a slow capacity fade?</summary>

Because of what causes it. Capacity fade is mostly lithium inventory quietly disappearing into a
film. Self-discharge climbs when the **separator** is being compromised — by swelling pressure, by
dendritic penetration, or by local overheating that melts and thins it. That is the same physical
path that ends in an internal short, so it is a warning about safety, not just about range.
</details>

<details><summary><b>5.</b> Your pack lives in a hot climate and is fast-charged. You can fix one thing. Which?</summary>

Cooling, almost certainly — but check it in section 9 rather than trusting the intuition. Set *cold /
fast charge* and read the window: the model puts the best temperature for that duty in the low
twenties, so a pack running at 45 °C is on the steep chemical side of the U and has the most to gain.
For a pack that is already cold, the answer flips: pre-heat before charging, don't cool.
</details>

## 12 · References

The narrative follows the *normal aging* treatment in the course text. Everything else is drawn from
the reviews and primary papers below — each section names the two or three it leans on.

**Course text**

1. G. L. Plett. *Battery Management Systems, Volume I: Battery Modeling.* Artech House, 2015.
   ISBN 978-1-63081-023-8.

**Mechanism taxonomy and overviews**

2. J. Vetter, P. Novák, M. R. Wagner, C. Veit, K.-C. Möller, J. O. Besenhard, M. Winter,
   M. Wohlfahrt-Mehrens, C. Vogler, A. Hammouche. Ageing mechanisms in lithium-ion batteries.
   *Journal of Power Sources* **147**, 269–281 (2005).
   [doi:10.1016/j.jpowsour.2005.01.006](https://doi.org/10.1016/j.jpowsour.2005.01.006)
3. M. Broussely, Ph. Biensan, F. Bonhomme, Ph. Blanchard, S. Herreyre, K. Nechev, R. J. Staniewicz.
   Main aging mechanisms in Li ion batteries. *Journal of Power Sources* **146**, 90–96 (2005).
   [doi:10.1016/j.jpowsour.2005.03.172](https://doi.org/10.1016/j.jpowsour.2005.03.172)
4. X. Han, L. Lu, Y. Zheng, X. Feng, Z. Li, J. Li, M. Ouyang. A review on the key issues of the
   lithium ion battery degradation among the whole life cycle. *eTransportation* **1**, 100005 (2019).
   [doi:10.1016/j.etran.2019.100005](https://doi.org/10.1016/j.etran.2019.100005)
5. J. S. Edge, S. O'Kane, R. Prosser, N. D. Kirkaldy, A. N. Patel, A. Hales, A. Ghosh, W. Ai, J. Chen,
   J. Yang, S. Li, M.-C. Pang, L. Bravo Diaz, A. Tomaszewska, M. W. Marzook, K. N. Radhakrishnan,
   H. Wang, Y. Patel, B. Wu, G. J. Offer. Lithium ion battery degradation: what you need to know.
   *Physical Chemistry Chemical Physics* **23**, 8200–8221 (2021).
   [doi:10.1039/D1CP00359C](https://doi.org/10.1039/D1CP00359C)

**Individual mechanisms**

6. M. B. Pinson, M. Z. Bazant. Theory of SEI formation in rechargeable batteries: capacity fade,
   accelerated aging and lifetime prediction. *Journal of The Electrochemical Society* **160**,
   A243–A250 (2013). [doi:10.1149/2.044302jes](https://doi.org/10.1149/2.044302jes) — *the √t law used in §3*
7. B. Rowden, N. Garcia-Araez. A review of gas evolution in lithium ion batteries.
   *Energy Reports* **6**, 10–18 (2020).
   [doi:10.1016/j.egyr.2020.02.022](https://doi.org/10.1016/j.egyr.2020.02.022) — *§4*
8. T. Waldmann, B.-I. Hogg, M. Wohlfahrt-Mehrens. Li plating as unwanted side reaction in commercial
   Li-ion cells — a review. *Journal of Power Sources* **384**, 107–124 (2018).
   [doi:10.1016/j.jpowsour.2018.02.063](https://doi.org/10.1016/j.jpowsour.2018.02.063) — *§6*
9. J. Christensen, J. Newman. Stress generation and fracture in lithium insertion materials.
   *Journal of Solid State Electrochemistry* **10**, 293–319 (2006).
   [doi:10.1007/s10008-006-0095-1](https://doi.org/10.1007/s10008-006-0095-1) — *§7*

**Diagnosis, trajectories and temperature**

10. C. R. Birkl, M. R. Roberts, E. McTurk, P. G. Bruce, D. A. Howey. Degradation diagnostics for
    lithium ion cells. *Journal of Power Sources* **341**, 373–386 (2017).
    [doi:10.1016/j.jpowsour.2016.12.011](https://doi.org/10.1016/j.jpowsour.2016.12.011)
11. T. Waldmann, M. Wilka, M. Kasper, M. Fleischhammer, M. Wohlfahrt-Mehrens. Temperature dependent
    ageing mechanisms in lithium-ion batteries — a post-mortem study.
    *Journal of Power Sources* **262**, 129–135 (2014).
    [doi:10.1016/j.jpowsour.2014.03.112](https://doi.org/10.1016/j.jpowsour.2014.03.112) — *the U-shape in §9*
12. P. M. Attia, A. Bills, F. Brosa Planella, P. Dechent, G. dos Reis, M. Dubarry, P. Gasper,
    R. Gilchrist, S. Greenbank, D. Howey, O. Liu, E. Khoo, Y. Preger, A. Soni, S. Sripad,
    A. G. Stefanopoulou, V. Sulzer. Review — "Knees" in lithium-ion battery aging trajectories.
    *Journal of The Electrochemical Society* **169**, 060517 (2022).
    [doi:10.1149/1945-7111/ac6d13](https://doi.org/10.1149/1945-7111/ac6d13)

**Schematic geometry**

13. Argonne National Laboratory, [Battery fundamentals](https://www.anl.gov/science-101/batteries).
14. NREL, [SEI, plating and cracking degradation models](https://www.osti.gov/servlets/purl/1834314).
15. DOE / Idaho National Laboratory, [Degradation during fast charge](https://www.osti.gov/servlets/purl/1855577).

---

*A note on what the numbers in this notebook are. The equations are teaching models fitted to give
the right direction, shape and rough relative magnitude — a √t film, Arrhenius factors, one-line
stress terms. The papers above are where the real parameters, the real spread between chemistries,
and the real uncertainty live. Do not quote a number from this notebook in coursework; quote the
paper it came from.*